# Supervised Learning Pipeline — Water Potability Dataset

**Prerequisite (Foundational)** — evaluation metrics reused in Experiments 6, 9 & 10.

This notebook covers:
1. Loading the dataset (via `kagglehub`)
2. Basic cleaning (handling missing values)
3. Train–test split
4. Training a simple classifier (scikit-learn)
5. Evaluation — accuracy, precision, recall, F1-score, confusion matrix


## 1. Imports

In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', 50)

RANDOM_STATE = 42


## 2. Download and Load the Dataset

In [ ]:
# Download latest version of the dataset from Kaggle
path = kagglehub.dataset_download("developerghost/water-potability")
print("Path to dataset files:", path)

# List files in the downloaded folder
files = os.listdir(path)
print("Files found:", files)


In [ ]:
# Load the first CSV file found in the dataset folder
csv_files = [f for f in files if f.endswith('.csv')]
print("CSV files:", csv_files)

file_path = os.path.join(path, csv_files[0])
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()


## 3. Quick Data Overview

In [ ]:
df.info()


In [ ]:
print("Missing values per column:\n")
print(df.isnull().sum())


## 4. Identify Target Column

The Water Potability dataset uses `Potability` (0 = not potable, 1 = potable) as the target.
The cell below falls back to the last column if the expected name isn't found, so this
notebook keeps working even if column naming differs slightly.


In [ ]:
target_col = 'Potability' if 'Potability' in df.columns else df.columns[-1]
print("Target column:", target_col)

df[target_col].value_counts()


In [ ]:
plt.figure(figsize=(5, 4))
df[target_col].value_counts().plot(kind='bar', color=['steelblue', 'darkorange'], edgecolor='black')
plt.title(f"Class Distribution — {target_col}")
plt.xlabel(target_col)
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Handle Missing Values

In [ ]:
# Impute missing numeric values with the column median (fit only on training data later
# to avoid leakage, but for simplicity here we impute globally with medians computed
# from the full feature set before splitting — a common, simple baseline approach)
feature_cols = [c for c in df.columns if c != target_col]

for col in feature_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Remaining missing values:", df.isnull().sum().sum())


## 6. Train–Test Split

In [ ]:
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 8. Train a Simple Classifier

In [ ]:
# Random Forest is a strong, easy-to-use baseline classifier
clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)


## 9. Evaluation Metrics

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")


In [ ]:
print("Full classification report:\n")
print(classification_report(y_test, y_pred))


## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Potable', 'Potable'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 11. Feature Importance (bonus)

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='seagreen', edgecolor='black')
plt.title("Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 12. Conclusion

- Loaded the Water Potability dataset directly from Kaggle using `kagglehub`.
- Handled missing values via median imputation.
- Split the data into training and test sets (80/20, stratified on the target).
- Scaled features and trained a `RandomForestClassifier` as a simple baseline model.
- Evaluated the model using **accuracy, precision, recall, F1-score**, and a **confusion matrix**.
- Inspected feature importances to understand which water-quality measurements matter most.

This evaluation workflow (train–test split → fit → accuracy/precision/recall/F1/confusion matrix) is the
foundational pattern reused in later experiments (Expts. 6, 9 & 10).
